In [ ]:
import matplotlib.pyplot as plt
from src.clustering import FactorClusterer
from src.data_loader import fetch_equity_returns
from src.pca_model import PCAFactorModel

# 1. Load Data and Extract Systematic PCA Loadings
tickers = [
    "AAPL",
    "MSFT",
    "GOOGL",
    "AMZN",
    "NVDA",
    "META",
    "TSLA",
    "AMD",
    "INTC",
    "QCOM",
    "JPM",
    "BAC",
    "WFC",
    "C",
    "GS",
    "XOM",
    "CVX",
    "COP",
    "SLB",
    "EOG",
]
returns = fetch_equity_returns(tickers, start_date="2023-01-01", end_date="2025-01-01")
pca = PCAFactorModel(n_components=5).fit(returns)

# 2. Cluster using Parametric UMAP + DBSCAN
clusterer = FactorClusterer(
    n_components=2, eps=0.35, min_samples=2, metric="cosine"
)
clusterer.fit(pca.factor_loadings)
clusters = clusterer.get_clusters()

# 3. Print Clustered Cohorts
print("=== Clustered Universe via Parametric UMAP + DBSCAN ===")
for cid, members in clusters.items():
    status = "Noise / Outlier" if cid == -1 else f"Cluster {cid}"
    print(f"{status:<16} ({len(members)} assets): {', '.join(members)}")

# 4. Visualize Latent Manifold & Cluster Boundaries
df_plot = clusterer.clustered_assets_
plt.figure(figsize=(10, 6))

scatter = plt.scatter(
    df_plot["umap_dim1"],
    df_plot["umap_dim2"],
    c=df_plot["cluster"],
    cmap="tab10",
    s=100,
    alpha=0.8,
    edgecolors="black",
)

for ticker in df_plot.index:
    plt.annotate(
        ticker,
        (df_plot.loc[ticker, "umap_dim1"], df_plot.loc[ticker, "umap_dim2"]),
        xytext=(5, 5),
        textcoords="offset points",
        fontsize=9,
        weight="bold",
    )

plt.colorbar(scatter, label="DBSCAN Cluster ID (-1 = Noise)")
plt.title(
    "Asset Clustering in Latent Space: Parametric UMAP + DBSCAN", fontsize=12
)
plt.xlabel("UMAP Dimension 1")
plt.ylabel("UMAP Dimension 2")
plt.grid(True, linestyle="--", alpha=0.5)
plt.show()